In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
import joblib
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("=== ENTRENAMIENTO RANDOM FOREST - COLDTRACK ===\n")

# Cargar datos procesados
X = pd.read_csv("X_features.csv")
y = pd.read_csv("y_target.csv").squeeze()   # convertir a Series

print(f"Dataset cargado: {X.shape[0]:,} registros × {X.shape[1]} features")

# ====================== 1. DIVISIÓN TEMPORAL (muy importante) ======================
# Usamos TimeSeriesSplit para respetar el orden cronológico
tscv = TimeSeriesSplit(n_splits=5)

print("Entrenando modelo con validación cruzada temporal...\n")

# ====================== 2. ENTRENAMIENTO DEL MODELO ======================
model = RandomForestClassifier(
    n_estimators=300,          # número de árboles
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',   # importante por el desbalanceo
    random_state=42,
    n_jobs=-1
)

# Entrenamos con el último split (train en datos antiguos, test en datos más recientes)
for train_index, test_index in tscv.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# Entrenamiento final
model.fit(X_train, y_train)

print("Modelo entrenado correctamente.\n")

# ====================== 3. EVALUACIÓN DEL MODELO ======================
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print("=== RESULTADOS DEL MODELO ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC AUC : {roc_auc_score(y_test, y_pred_proba):.4f}\n")

print("Reporte de clasificación:")
print(classification_report(y_test, y_pred, target_names=['Normal (0)', 'Mantenimiento (1)']))

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
print("\nMatriz de Confusión:")
print(cm)

# ====================== 4. IMPORTANCIA DE FEATURES ======================
importancia = pd.DataFrame({
    'Feature': X.columns,
    'Importancia': model.feature_importances_
}).sort_values('Importancia', ascending=False)

print("\nTop 10 features más importantes:")
print(importancia.head(10))

# ====================== 5. GUARDAR EL MODELO ======================
joblib.dump(model, "coldtrack_rf_model.pkl")
print("\n✅ Modelo guardado como: coldtrack_rf_model.pkl")

# Guardar también las columnas usadas (importante para inferencia futura)
joblib.dump(X.columns.tolist(), "feature_columns.pkl")
print("✅ Columnas guardadas como: feature_columns.pkl")


=== ENTRENAMIENTO RANDOM FOREST - COLDTRACK ===

Dataset cargado: 311,040 registros × 25 features
Entrenando modelo con validación cruzada temporal...

Modelo entrenado correctamente.

=== RESULTADOS DEL MODELO ===
Accuracy: 0.9999
ROC AUC : 1.0000

Reporte de clasificación:
                   precision    recall  f1-score   support

       Normal (0)       1.00      1.00      1.00     47722
Mantenimiento (1)       1.00      1.00      1.00      4118

         accuracy                           1.00     51840
        macro avg       1.00      1.00      1.00     51840
     weighted avg       1.00      1.00      1.00     51840


Matriz de Confusión:
[[47719     3]
 [    0  4118]]

Top 10 features más importantes:
                Feature  Importancia
4      ConsumoElectrico     0.235508
5             Vibration     0.201131
21  compressor_overwork     0.151018
19   critical_vibration     0.140990
1               InHumid     0.094030
6    temp_diff_setpoint     0.079779
0                inTe